In [1]:
import pandas as pd
from pathlib import Path

FILE_PATH = Path(
    r"C:\Users\KIIT\OneDrive\Desktop\Financial Analytics Project\Raw Data\documents.xlsx"
)

raw = pd.read_excel(
    FILE_PATH,
    sheet_name="Documents",
    header=1
)

print("Shape:", raw.shape)
print("\nColumns:")
print(raw.columns.tolist())

raw.head()

Shape: (1585, 4)

Columns:
['id', 'company_id', 'Year', 'Annual_Report']


,id,company_id,Year,Annual_Report
0,1,ABB,2024,https://www.bseindia.com/xml-data/corpfiling/A...
1,2,ABB,2023,https://www.bseindia.com/xml-data/corpfiling/A...
2,3,ABB,2022,https://www.bseindia.com/bseplus/AnnualReport/...
3,4,ABB,2021,https://www.bseindia.com/bseplus/AnnualReport/...
4,5,ABB,2020,https://www.bseindia.com/bseplus/AnnualReport/...


In [2]:
documents = raw.copy()

# Standardize column names
documents.columns = [
    "id",
    "company_id",
    "year",
    "annual_report"
]

# Clean text fields
documents["company_id"] = documents["company_id"].astype(str).str.strip()
documents["annual_report"] = documents["annual_report"].astype(str).str.strip()

# Convert year to numeric
documents["year"] = pd.to_numeric(documents["year"], errors="coerce")

print("Shape:", documents.shape)

print("\nMissing values:")
print(documents.isna().sum())

print("\nDuplicate IDs:", documents["id"].duplicated().sum())

print(
    "Duplicate company-year:",
    documents.duplicated(subset=["company_id", "year"]).sum()
)

print("\nYear range:")
print(documents["year"].min(), "to", documents["year"].max())

Shape: (1585, 4)

Missing values:
id               0
company_id       0
year             0
annual_report    0
dtype: int64

Duplicate IDs: 0
Duplicate company-year: 1

Year range:
2007 to 2024


In [3]:
print(
    "Missing report URLs:",
    documents["annual_report"].isna().sum()
)

print(
    "Invalid URL format:",
    (~documents["annual_report"].str.startswith(("http://", "https://"))).sum()
)

Missing report URLs: 0
Invalid URL format: 383


In [4]:
duplicates = documents[
    documents.duplicated(
        subset=["company_id", "year"],
        keep=False
    )
].sort_values(["company_id", "year"])

print("Duplicate company-year rows:", len(duplicates))

duplicates.head(20)

Duplicate company-year rows: 2


,id,company_id,year,annual_report
557,558,HAL,2011,nan
558,559,HAL,2011,https://www.bseindia.com/HIS_ANN_RPT/HISTANNR/...


In [5]:
# Sort so rows with a report URL come first
documents = documents.sort_values(
    by="annual_report",
    na_position="last"
)

# Keep one record per company-year
documents = documents.drop_duplicates(
    subset=["company_id", "year"],
    keep="first"
).reset_index(drop=True)

print("Shape after deduplication:", documents.shape)
print(
    "Duplicate company-year:",
    documents.duplicated(["company_id", "year"]).sum()
)

Shape after deduplication: (1584, 4)
Duplicate company-year: 0


In [6]:
invalid_urls = documents[
    ~documents["annual_report"].str.startswith(
        ("http://", "https://"),
        na=False
    )
]

print("Non-standard report URLs:", len(invalid_urls))

invalid_urls[["company_id", "year", "annual_report"]].head(20)

Non-standard report URLs: 382


,company_id,year,annual_report
0,KOTAKBANK,2011,5002470311.pdf
1,KOTAKBANK,2015,5002470315.pdf
2,KOTAKBANK,2016,5002470316.pdf
3,IRCTC,2017,Null
4,IRCTC,2009,Null
5,IRFC,2020,Null
6,IRFC,2019,Null
7,IRCTC,2010,Null
8,IRFC,2018,Null
9,IRFC,2016,Null


In [7]:
CLEANED_DATA = Path(
    r"C:\Users\KIIT\OneDrive\Desktop\Financial Analytics Project\Cleaned Data"
)

CLEANED_DATA.mkdir(parents=True, exist_ok=True)

documents.to_csv(
    CLEANED_DATA / "documents_clean.csv",
    index=False
)

print("Saved:", CLEANED_DATA / "documents_clean.csv")

Saved: C:\Users\KIIT\OneDrive\Desktop\Financial Analytics Project\Cleaned Data\documents_clean.csv


In [8]:
import pandas as pd

documents = pd.read_csv(
    r"C:\Users\KIIT\OneDrive\Desktop\Financial Analytics Project\Cleaned Data\documents_clean.csv"
)

print("Shape:", documents.shape)
print("Columns:", documents.columns.tolist())

print("\nMissing values:")
print(documents.isna().sum())

print("\nDuplicate IDs:", documents["id"].duplicated().sum())

print("\nUnique companies:", documents["company_id"].nunique())

Shape: (1584, 4)
Columns: ['id', 'company_id', 'year', 'annual_report']

Missing values:
id                0
company_id        0
year              0
annual_report    51
dtype: int64

Duplicate IDs: 0

Unique companies: 99
